In [11]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [12]:
data_ace_24 = pd.read_csv("Ace_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_24 = pd.read_csv("Discover_погружение_24часа.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_ace_af = pd.read_csv("Ace_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')
data_discover_af = pd.read_csv("Discover_погружение_AF.csv", sep=';', decimal=',', parse_dates=['datetime'], index_col='datetime')

In [16]:
def split_datasets_intersection(ace, disc, split_date, scenario):
    """
    Разделяет два датасета ace и discover на train/test по заданной дате split_date.
    При необходимости выравнивает временные индексы (пересечение) и удаляет строки с NaN.

    Сценарии:
    1. 'ace-ace'   — обучение и тестирование на ACE (train до split_date, test после)
    2. 'ace-disc'  — обучение на ACE (до split_date), тестирование на DISC (после split_date)
    3. 'disc-ace'  — обучение на DISC (до split_date), тестирование на ACE (после split_date)
    4. 'disc-disc' — обучение и тестирование на DISC (train до split_date, test после)
    """
    a = ace.copy()
    d = disc.copy()

    # Выравниваем временные индексы так, чтобы все наборы были на одинаковых данных в пересечениях
    common_idx = a.index.intersection(d.index)
    a = a.loc[common_idx].dropna(axis=0, how='any')
    d = d.loc[common_idx].dropna(axis=0, how='any')

    # Дата разделения
    split_date = pd.Timestamp(split_date)

    # Сценарии
    if scenario == 'ace-ace':
        train = a[a.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'ace-disc':
        train = a[a.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    elif scenario == 'disc-ace':
        train = d[d.index < split_date].copy()
        test  = a[a.index >= split_date].copy()

    elif scenario == 'disc-disc':
        train = d[d.index < split_date].copy()
        test  = d[d.index >= split_date].copy()

    else:
        raise ValueError(f"Некорректный сценарий '{scenario}'. "
                         f"Должен быть один из: 'ace-ace', 'ace-disc', 'disc-ace', 'disc-disc'")

    return train, test

In [38]:
def build_models(random_state=42):
    models = {}
    models['Linear'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lin", LinearRegression())
    ])
    
    models['Ridge'] = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=1.0,
            solver='auto',
            random_state=random_state))
    ])

    models['Lasso'] = Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(
            alpha=0.0005,
            tol=0.01,
            max_iter=500, 
            random_state=random_state))
    ])
    
    models['LGBM'] = Pipeline([
        ("boost", LGBMRegressor(
        n_estimators=700, # 1500, остальное дефолтное
        learning_rate=0.02,
        max_depth=-1, #3
        num_leaves=30,
        subsample=0.9,
        colsample_bytree=0.8,
        early_stopping_rounds=50, # оставить
        random_state=random_state,
        verbose=-1))
    ])
    
    models['MLP'] = Pipeline([ # возможно реализация модели кривая
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
        hidden_layer_sizes=(256, 256, 128),
        activation='tanh', # relu
        solver='adam',
        alpha=1e-4,
        learning_rate_init=1e-3,
        early_stopping=True,
        validation_fraction=0.1,
        max_iter=1500,
        random_state=random_state))
    ])
    return models
    
models = build_models()

def evaluate_models(ace, disc, split_date, all_targets, target, scenario='ace-ace', delays='24h', results_list=None):
    """
    Обучает все модели на train/test наборах, полученных из split_datasets_intersection(),
    и вычисляет метрики mse и r2_score

    Выход:
    results : dict
        Метрики по каждой модели: {model_name: {'mse': ..., 'r2': ...}}
    """
    # Разделяем данные по сценарию
    train, test = split_datasets_intersection(ace, disc, split_date, scenario=scenario)
        
    # Определение признаков по заданным таргетам (здесь убираю из признаков все значения dst+1, dst+2, dst+3)
    feature_cols = [c for c in train.columns if c not in all_targets]

    train = train.dropna(subset=feature_cols + [target])
    test = test.dropna(subset=feature_cols + [target])

    X_train, y_train = train[feature_cols].values, train[target].values
    X_test, y_test = test[feature_cols].values, test[target].values

    if results_list is None:
        results_list = []

    print(f"\n=== {scenario.upper()} ===")
    for name, model in models.items():
        try:
            if name == 'LGBM':
                # Для бустинга выделяем валидационный набор
                X_tr, X_val, y_tr, y_val = train_test_split(
                    X_train, y_train, test_size=0.1, random_state=42)
                model.fit(
                    X_tr, y_tr,
                    boost__eval_set=[(X_val, y_val)],
                    boost__eval_metric='l2')
            else:
                model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            results_list.append({
                'Target': target,
                'Delays': delays,
                'Scenario': scenario,
                'Forecast_model': name,
                'RMSE': rmse,
                'R2': r2
            })

            print(f"{name}: rmse={rmse:.4f}, r2={r2:.4f}")

        except Exception as e:
            print(f"{name:10s} -> Ошибка обучения: {e}")

    return results_list

In [33]:
def models_shuffle(ace_df, disc_df, split_date, all_targets, target, scenario='suffle', delays='24h', results_list=None):
    """
    Объединяет данные ace и discover, добавляет признак is_ace,
    разделяет на train/test по дате и обучает модели.
    """
    ace = ace_df.sort_index()
    disc = disc_df.sort_index()

    split_date = pd.Timestamp(split_date)
    
    ace['is_ace'] = 1
    disc['is_ace'] = 0

    ace_train = ace.loc[:split_date]
    ace_test = ace.loc[split_date:]
    
    disc_train = disc.loc[:split_date]
    disc_test = disc.loc[split_date:]

    split_date = pd.Timestamp(split_date)
    
    combined_train = pd.concat([ace_train, disc_train], ignore_index=False)
    combined_train = combined_train.sort_values('datetime')
    
    # train = combined_data[combined_data.index < split_date].copy()
    # test = combined_data[combined_data.index >= split_date].copy()
    
    feature_cols = [c for c in ace_train.columns if c not in all_targets]
    
    train = combined_train.dropna(subset=feature_cols + [target])
    test = disc_test.dropna(subset=feature_cols + [target])

    X_train, y_train = train[feature_cols].values, train[target].values
    X_test, y_test = test[feature_cols].values, test[target].values
    
    if results_list is None:
        results_list = []

    print(f"\n=== {scenario.upper()} ===")
    for name, model in models.items():
        try:
            if name == 'LGBM':
                # Для бустинга выделяем валидационный набор
                X_tr, X_val, y_tr, y_val = train_test_split(
                    X_train, y_train, test_size=0.1, random_state=42)
                model.fit(
                    X_tr, y_tr,
                    boost__eval_set=[(X_val, y_val)],
                    boost__eval_metric='l2')
            else:
                model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
                   
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            r2 = r2_score(y_test, y_pred)

            results_list.append({
                'Target': target,
                'Delays': delays,
                'Scenario': scenario,
                'Forecast_model': name,
                'RMSE': rmse,
                'R2': r2
            })

            print(f"{name}: rmse={rmse:.4f}, r2={r2:.4f}")

        except Exception as e:
            print(f"{name:10s} -> Ошибка обучения: {e}")

    return results_list

In [23]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_24_copy = data_ace_24.copy()
data_discover_24_copy = data_discover_24.copy()
data_ace_af_copy = data_ace_af.copy()
data_discover_af_copy = data_discover_af.copy()

# 1. Задание переменных для обучения
split_date = "2021-01-01" # дата по которой происходит разбиение данных на тренировочный и тестовый наборы
scenarios = ['ace-ace', 'ace-disc', 'disc-ace', 'disc-disc']
targets = ['Dst_plus1', 'Dst_plus2', 'Dst_plus3']

results_list = []

In [25]:
# 2. Обучение для датасетов с глубиной - 24 часа по всем переменным
print(f"\n==== Depth - 24h ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    for scen in scenarios:
        res = evaluate_models(data_ace_24_copy, data_discover_24_copy, split_date, targets, target_col, scenario=scen, delays='24h', results_list=results_list)


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====

=== ACE-ACE ===
Linear: rmse=3.2726, r2=0.9635
Ridge: rmse=3.2729, r2=0.9635
Lasso: rmse=3.2732, r2=0.9635
LGBM: rmse=3.8413, r2=0.9497
MLP: rmse=6.4018, r2=0.8604

=== ACE-DISC ===
Linear: rmse=3.4078, r2=0.9604
Ridge: rmse=3.4074, r2=0.9605
Lasso: rmse=3.3910, r2=0.9608
LGBM: rmse=3.8603, r2=0.9492
MLP: rmse=7.5358, r2=0.8066

=== DISC-ACE ===
Linear: rmse=3.2817, r2=0.9633
Ridge: rmse=3.2819, r2=0.9633
Lasso: rmse=3.2823, r2=0.9633
LGBM: rmse=3.9258, r2=0.9475
MLP: rmse=6.5563, r2=0.8536

=== DISC-DISC ===
Linear: rmse=3.2303, r2=0.9645
Ridge: rmse=3.2304, r2=0.9645
Lasso: rmse=3.2302, r2=0.9645
LGBM: rmse=3.8000, r2=0.9508
MLP: rmse=6.5357, r2=0.8545

==== Forecast of DST_PLUS2 ====

=== ACE-ACE ===
Linear: rmse=5.1485, r2=0.9097
Ridge: rmse=5.1486, r2=0.9097
Lasso: rmse=5.1474, r2=0.9097
LGBM: rmse=5.4886, r2=0.8974
MLP: rmse=8.2947, r2=0.7656

=== ACE-DISC ===
Linear: rmse=5.2954, r2=0.9045
Ridge: rmse=5.2953, r2=0.9045
Las

In [29]:
# 3. Обучение для датасетов с глубиной, заданной автокорреляционной функцией
print(f"\n==== Depth - autocorrelation function ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    for scen in scenarios:
        res = evaluate_models(data_ace_af_copy, data_discover_af_copy, split_date, targets, target_col, scenario=scen, delays='auto_func', results_list=results_list)


==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====

=== ACE-ACE ===
Linear: rmse=3.2756, r2=0.9633
Ridge: rmse=3.2758, r2=0.9633
Lasso: rmse=3.2753, r2=0.9633
LGBM: rmse=3.8609, r2=0.9490
MLP: rmse=5.7900, r2=0.8854

=== ACE-DISC ===
Linear: rmse=3.4096, r2=0.9603
Ridge: rmse=3.4092, r2=0.9603
Lasso: rmse=3.3921, r2=0.9607
LGBM: rmse=3.8918, r2=0.9482
MLP: rmse=6.9425, r2=0.8353

=== DISC-ACE ===
Linear: rmse=3.2886, r2=0.9630
Ridge: rmse=3.2888, r2=0.9630
Lasso: rmse=3.2894, r2=0.9630
LGBM: rmse=3.9751, r2=0.9460
MLP: rmse=5.8104, r2=0.8846

=== DISC-DISC ===
Linear: rmse=3.2337, r2=0.9643
Ridge: rmse=3.2338, r2=0.9643
Lasso: rmse=3.2330, r2=0.9643
LGBM: rmse=3.8459, r2=0.9494
MLP: rmse=5.7701, r2=0.8862

==== Forecast of DST_PLUS2 ====

=== ACE-ACE ===
Linear: rmse=5.1594, r2=0.9090
Ridge: rmse=5.1595, r2=0.9090
Lasso: rmse=5.1580, r2=0.9091
LGBM: rmse=5.4517, r2=0.8984
MLP: rmse=8.2472, r2=0.7675

=== ACE-DISC ===
Linear: rmse=5.2922, r2=0.9043
Ridge: rmse=

In [34]:
# 4. Предсказание на перемешанных данных с признаком is_ace
print(f"\n==== Depth - 24h ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = models_shuffle(data_ace_24_copy, data_discover_24_copy, split_date, targets, target_col, scenario='suffle', delays='24h', results_list=results_list)

print(f"\n==== Depth - autocorrelation function ====")
for target_col in targets:
    print(f"\n==== Forecast of {target_col.upper()} ====")
    res = models_shuffle(data_ace_af_copy, data_discover_af_copy, split_date, targets, target_col, scenario='shuffle', delays='auto_func', results_list=results_list)


==== Depth - 24h ====

==== Forecast of DST_PLUS1 ====

=== SUFFLE ===
Linear: rmse=3.2303, r2=0.9645
Ridge: rmse=3.2302, r2=0.9645
Lasso: rmse=3.2259, r2=0.9646
LGBM: rmse=3.2513, r2=0.9640
MLP: rmse=4.1641, r2=0.9409

==== Forecast of DST_PLUS2 ====

=== SUFFLE ===
Linear: rmse=5.0381, r2=0.9135
Ridge: rmse=5.0380, r2=0.9135
Lasso: rmse=5.0342, r2=0.9137
LGBM: rmse=4.8478, r2=0.9199
MLP: rmse=6.4076, r2=0.8601

==== Forecast of DST_PLUS3 ====

=== SUFFLE ===
Linear: rmse=6.4449, r2=0.8584
Ridge: rmse=6.4449, r2=0.8584
Lasso: rmse=6.4433, r2=0.8585
LGBM: rmse=6.2022, r2=0.8689
MLP: rmse=8.1329, r2=0.7746

==== Depth - autocorrelation function ====

==== Forecast of DST_PLUS1 ====

=== SHUFFLE ===
Linear: rmse=3.2410, r2=0.9641
Ridge: rmse=3.2410, r2=0.9641
Lasso: rmse=3.2366, r2=0.9642
LGBM: rmse=3.2856, r2=0.9631
MLP: rmse=4.0646, r2=0.9435

==== Forecast of DST_PLUS2 ====

=== SHUFFLE ===
Linear: rmse=5.0689, r2=0.9122
Ridge: rmse=5.0689, r2=0.9122
Lasso: rmse=5.0652, r2=0.9123
LGB

In [35]:
results_df = pd.DataFrame(results_list)

In [36]:
results_df

,Target,Delays,Scenario,Forecast_model,RMSE,R2
0,Dst_plus1,24h,ace-ace,Linear,3.272603,0.963524
1,Dst_plus1,24h,ace-ace,Ridge,3.272857,0.963518
2,Dst_plus1,24h,ace-ace,Lasso,3.273231,0.963510
3,Dst_plus1,24h,ace-ace,LGBM,3.841332,0.949744
4,Dst_plus1,24h,ace-ace,MLP,6.401839,0.860418
...,...,...,...,...,...,...
155,Dst_plus3,auto_func,shuffle,Linear,6.484136,0.856271
156,Dst_plus3,auto_func,shuffle,Ridge,6.484121,0.856272
157,Dst_plus3,auto_func,shuffle,Lasso,6.481531,0.856387
158,Dst_plus3,auto_func,shuffle,LGBM,6.204708,0.868392


In [37]:
results_df.to_excel("results/models-no-adaptation.xlsx", index=False)